<a href="https://colab.research.google.com/github/asegura4488/MecanicaClasica/blob/main/Cono_velocidades_angulares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cono rodando sin deslizar · Animación 3D interactiva

Este notebook genera una animación **Plotly** que puedes reproducir directamente en **Google Colab**, y exporta un **HTML autónomo** con controles de reproducción.

Convención de ejes (laboratorio):

\[
\hat n=(\cos\alpha\cos\phi,\ \cos\alpha\sin\phi,\ \sin\alpha),\qquad
\boldsymbol\omega_{\rm propia}=\dot\psi\hat n,\qquad
\boldsymbol\omega_{\rm prec}=\dot\phi\hat k.
\]

Para rodadura sin deslizamiento, con vértice fijo:

\[
\boxed{\dot\phi=-\dot\psi\sin\alpha},\qquad
\boxed{\boldsymbol\omega_{\rm total}=\dot\psi\cos\alpha(\cos\phi,\sin\phi,0)}.
\]

La vista izquierda incluye el cono, su generatriz de contacto y una **línea material** que gira sobre la superficie; la derecha muestra la suma vectorial, y abajo aparecen las componentes instantáneas de cada vector.

**Uso:** Ejecuta las celdas en orden. Cambia `ALPHA_GRADOS` y `PSI_DOT` en la segunda celda y vuelve a ejecutar para explorar otros valores. La última celda permite descargar el HTML desde Colab.

In [ ]:
#@title 2. Parámetros del cono y de la animación
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import sys

ALPHA_GRADOS = 30.0   #@param {type:"number"}
PSI_DOT = 2.0         #@param {type:"number"}
N_FRAMES = 60         #@param {type:"integer"}
LONGITUD_GENERATRIZ = 1.45  #@param {type:"number"}
ARCHIVO_HTML = "cono_velocidades_colab.html"

assert 0 < ALPHA_GRADOS < 90, "Usa 0 < alpha < 90 grados"
assert PSI_DOT != 0, "La velocidad propia debe ser distinta de cero"
assert 20 <= N_FRAMES <= 180, "Usa entre 20 y 180 fotogramas"
alpha = np.deg2rad(ALPHA_GRADOS)
phi_dot = -PSI_DOT * np.sin(alpha)  # rodadura sin deslizamiento
periodo = 2 * np.pi / abs(phi_dot)
tiempos = np.linspace(0, periodo, N_FRAMES, endpoint=False)

print(f"alpha={ALPHA_GRADOS:.1f} grados | dot(psi)={PSI_DOT:.3f} rad/s | dot(phi)={phi_dot:.3f} rad/s")
print("Relación: dot(phi) = -dot(psi) sin(alpha)")

In [ ]:
#@title 3. Geometría y vectores en el laboratorio
# e_r y e_phi son las direcciones radial y azimutal del laboratorio.
# n es el eje del cono; e1,e2 generan su plano transversal.
def geometria(t):
    phi = phi_dot * t
    psi = PSI_DOT * t
    er = np.array([np.cos(phi), np.sin(phi), 0.0])
    et = np.array([-np.sin(phi), np.cos(phi), 0.0])
    k = np.array([0.0, 0.0, 1.0])
    n = np.cos(alpha)*er + np.sin(alpha)*k
    e1 = np.sin(alpha)*er - np.cos(alpha)*k
    e2 = et
    g = er  # generatriz horizontal que toca el suelo

    # Velocidades angulares en coordenadas del laboratorio
    w_psi = PSI_DOT*n
    w_phi = phi_dot*k
    w_total = w_psi+w_phi

    # Superficie de cono circular: semivértice alpha, vértice en O=(0,0,0)
    beta = np.linspace(0, 2*np.pi, 45)
    s = np.linspace(0.04, LONGITUD_GENERATRIZ, 12)
    B, S = np.meshgrid(beta, s)
    dirs = np.cos(alpha)*n[:,None,None] + np.sin(alpha)*(
        np.cos(B)[None,:,:]*e1[:,None,None] + np.sin(B)[None,:,:]*e2[:,None,None]
    )
    xyz = S[None,:,:]*dirs

    # Base circular
    rim_dirs = np.cos(alpha)*n[:,None] + np.sin(alpha)*(
        np.cos(beta)[None,:]*e1[:,None] + np.sin(beta)[None,:]*e2[:,None]
    )
    rim = LONGITUD_GENERATRIZ*rim_dirs
    # Línea material: un radio pintado sobre la superficie que gira con psi.
    marca_dir = np.cos(alpha)*n + np.sin(alpha)*(
        np.cos(psi)*e1 + np.sin(psi)*e2
    )
    marca = np.outer(np.linspace(0,LONGITUD_GENERATRIZ,15),marca_dir)
    contacto = np.outer(np.linspace(0,LONGITUD_GENERATRIZ,15),g)
    return dict(phi=phi,psi=psi,n=n,g=g,wp=w_psi,wf=w_phi,wt=w_total,
                xyz=xyz, rim=rim,marca=marca,contacto=contacto)

# Validaciones geométricas y cinemáticas
estado = geometria(0.42)
assert np.allclose(np.linalg.norm(estado['n']),1)
assert np.allclose(estado['wt'][2],0)
assert np.allclose(np.cross(estado['wt'],estado['g']),np.zeros(3))
assert np.min(estado['xyz'][2]) >= -1e-10
print('Comprobaciones correctas: la superficie no penetra el plano y ω_total es horizontal.')

In [ ]:
#@title 4. Construir y mostrar animación interactiva + exportar HTML
COLORES = {"n":"#208B56","wp":"#E59A22","wf":"#337AB7","wt":"#D7394B","g":"#7560AF"}

# Escala gráfica única para las velocidades: direcciones y proporciones correctas.
ESCALA_W = 0.68

def vec_line(v, color, name, scale=1.0, showlegend=True):
    tip = scale*np.asarray(v)
    return go.Scatter3d(x=[0,tip[0]],y=[0,tip[1]],z=[0,tip[2]],
                        mode='lines+markers', name=name,showlegend=showlegend,
                        line=dict(color=color,width=10),
                        marker=dict(color=color,size=[2,5]), hoverinfo='name')

def vec_tip(v,color,scale=1.0):
    p = scale*np.asarray(v)
    mag = np.linalg.norm(p)
    unit = p/(mag if mag else 1)
    return go.Cone(x=[p[0]],y=[p[1]],z=[p[2]],
                   u=[unit[0]],v=[unit[1]],w=[unit[2]],
                   anchor='tip',sizemode='absolute',sizeref=0.16,showscale=False,
                   colorscale=[[0,color],[1,color]],hoverinfo='skip',showlegend=False)

def tabla_textos(e):
    vectores=[e['n'],e['wp'],e['wf'],e['wt'],e['g']]
    return [[etiqueta,*(f'{q:+.3f}' for q in v),f'{np.linalg.norm(v):.3f}']
            for etiqueta,v in zip(['n (eje)','dot(ψ) n','dot(φ) k','ω total','g (contacto)'],vectores)]

def traces_dinamicos(e):
    xyz=e['xyz']; rim=e['rim']; contacto=e['contacto']; marca=e['marca']
    izq=[
        go.Surface(x=xyz[0],y=xyz[1],z=xyz[2],opacity=0.25,
                   showscale=False,colorscale=[[0,'#76b8ce'],[1,'#76b8ce']],
                   hoverinfo='skip',name='Superficie del cono'),
        go.Scatter3d(x=rim[0],y=rim[1],z=rim[2],mode='lines',
                     line=dict(color='#387795',width=5),showlegend=False,hoverinfo='skip'),
        go.Scatter3d(x=contacto[:,0],y=contacto[:,1],z=contacto[:,2],
                     mode='lines',line=dict(color=COLORES['g'],width=9),
                     name='Generatriz de contacto',showlegend=True),
        go.Scatter3d(x=marca[:,0],y=marca[:,1],z=marca[:,2],
                     mode='lines',line=dict(color='#9D59B3',width=6),
                     showlegend=True,name='Línea material (giro ψ)'),
        vec_line(e['n'],COLORES['n'],'n (eje del cono)',0.9),
        vec_tip(e['n'],COLORES['n'],0.9)
    ]
    der=[]
    for vec,col,lab,esc in [
        (e['n'],COLORES['n'],'n (unitario)',0.85),
        (e['wp'],COLORES['wp'],'ω propia = ψ̇ n',ESCALA_W),
        (e['wf'],COLORES['wf'],'ω precesión = φ̇ k',ESCALA_W),
        (e['wt'],COLORES['wt'],'ω total',ESCALA_W)]:
        der.extend([vec_line(vec,col,lab,esc),vec_tip(vec,col,esc)])
    # El paralelogramo de velocidades: ωψ + ωφ = ωtotal.
    p=ESCALA_W*e['wp']; q=ESCALA_W*e['wf']; total=p+q
    der.extend([
        go.Scatter3d(x=[p[0],total[0]],y=[p[1],total[1]],z=[p[2],total[2]],
                     mode='lines',line=dict(color=COLORES['wf'],dash='dash',width=4),
                     showlegend=False,hoverinfo='skip'),
        go.Scatter3d(x=[q[0],total[0]],y=[q[1],total[1]],z=[q[2],total[2]],
                     mode='lines',line=dict(color=COLORES['wp'],dash='dash',width=4),
                     showlegend=False,hoverinfo='skip')
    ])
    filas=tabla_textos(e)
    labels=[]
    for row,color in zip(filas,[COLORES[k] for k in ('n','wp','wf','wt','g')]):
        labels.append(go.Scatter(
            x=[0,1.1,2.2,3.3,4.4],y=[0]*5,text=row,mode='text',
            textfont=dict(size=13,color=color),showlegend=False,hoverinfo='skip'))
    return izq,der,labels

fig=make_subplots(rows=2,cols=2,
    specs=[[{'type':'scene'},{'type':'scene'}],[{'type':'xy','colspan':2},None]],
    row_heights=[0.76,0.24],column_widths=[0.51,0.49],
    vertical_spacing=0.085,horizontal_spacing=0.05,
    subplot_titles=['Cono y generatriz de contacto','Suma vectorial de velocidades','Componentes cartesianas en el laboratorio'])

# Plano horizontal fijo para situar el cono
xp,yp=np.meshgrid(np.linspace(-1.65,1.65,2),np.linspace(-1.65,1.65,2))
fig.add_trace(go.Surface(x=xp,y=yp,z=np.zeros_like(xp)-0.008,
    opacity=0.12,colorscale=[[0,'#A1A9AE'],[1,'#A1A9AE']],
    showscale=False,hoverinfo='skip',name='Plano horizontal'),row=1,col=1)

primero=geometria(tiempos[0]); izq,der,filas=traces_dinamicos(primero)
indices=[]
for tr in izq:
    fig.add_trace(tr,row=1,col=1)
    indices.append(len(fig.data)-1)
for tr in der:
    fig.add_trace(tr,row=1,col=2)
    indices.append(len(fig.data)-1)

# Encabezados de la tabla de componentes
fig.add_trace(go.Scatter(x=[0,1.1,2.2,3.3,4.4],y=[5.5]*5,
    text=['Vector','Componente x','Componente y','Componente z','Magnitud'],
    mode='text',textfont=dict(color='#25344E',size=13),
    hoverinfo='skip',showlegend=False),row=2,col=1)
for j,tr in enumerate(filas):
    tr.y=[4.7-j*0.9]*5
    fig.add_trace(tr,row=2,col=1)
    indices.append(len(fig.data)-1)

# Indicador de tiempo, phi y psi dentro de la tabla
fig.add_trace(go.Scatter(x=[2.2],y=[-0.18],
    text=['t=0.00 s | φ=0.0° | ψ=0.0°'],mode='text',
    textfont=dict(size=12,color='#6B7280'),hoverinfo='skip',showlegend=False),row=2,col=1)
indices.append(len(fig.data)-1)

# Fotogramas: modificamos superficies, vectores y textos
frames=[]
for t_idx,t in enumerate(tiempos):
    e=geometria(t)
    izq,der,filas=traces_dinamicos(e)
    for j,tr in enumerate(filas):
        tr.y=[4.7-j*0.9]*5
    estado_texto=go.Scatter(x=[2.2],y=[-0.18],
        text=[f't={t:.2f} s | φ={np.rad2deg(e["phi"]):+.1f}° | ψ={np.rad2deg(e["psi"]):+.1f}°'],
        mode='text',textfont=dict(size=12,color='#6B7280'))
    frames.append(go.Frame(name=str(t_idx),data=izq+der+filas+[estado_texto],traces=indices))
fig.frames=frames

camara=dict(eye=dict(x=1.55,y=1.80,z=1.30))
fig.update_scenes(xaxis=dict(title='x',range=[-1.65,1.65],showbackground=False),
                  yaxis=dict(title='y',range=[-1.65,1.65],showbackground=False),
                  zaxis=dict(title='z',range=[-0.13,1.55],showbackground=False),
                  aspectmode='manual',aspectratio=dict(x=1,y=1,z=0.65),
                  camera=camara,row=1,col=1)
fig.update_scenes(xaxis=dict(title='ωx',range=[-1.6,1.6],showbackground=False),
                  yaxis=dict(title='ωy',range=[-1.6,1.6],showbackground=False),
                  zaxis=dict(title='ωz',range=[-1.05,1.6],showbackground=False),
                  aspectmode='cube',camera=dict(eye=dict(x=1.5,y=1.65,z=1.15)),row=1,col=2)
fig.update_xaxes(range=[-0.65,5.0],visible=False,row=2,col=1)
fig.update_yaxes(range=[-0.5,6.1],visible=False,row=2,col=1)
fig.update_layout(
    title=dict(text=f'Cono sin deslizamiento · α={ALPHA_GRADOS:g}° · ψ̇={PSI_DOT:g} rad/s · φ̇={phi_dot:.3f} rad/s',x=0.5),
    width=1250,height=830,margin=dict(l=5,r=5,t=100,b=20),
    paper_bgcolor='white',plot_bgcolor='white',
    legend=dict(orientation='h',x=0.5,xanchor='center',y=1.05,yanchor='bottom',font=dict(size=11)),
    updatemenus=[dict(type='buttons',showactive=False,x=0.40,y=0.02,xanchor='right',yanchor='top',
        buttons=[
            dict(label='▶ Reproducir',method='animate',
                 args=[None,dict(frame=dict(duration=90,redraw=True),transition=dict(duration=0),
                                 fromcurrent=True,mode='immediate')]),
            dict(label='⏸ Pausar',method='animate',
                 args=[[None],dict(frame=dict(duration=0,redraw=False),
                                   transition=dict(duration=0),mode='immediate')])])],
    sliders=[dict(active=0,x=0.43,y=0.02,len=0.55,yanchor='top',
        currentvalue=dict(prefix='Fotograma: ',font=dict(size=11)),
        steps=[dict(label=str(i),method='animate',
                    args=[[str(i)],dict(mode='immediate',frame=dict(duration=0,redraw=True),
                                        transition=dict(duration=0))]) for i in range(N_FRAMES)])]
)

fig.write_html(ARCHIVO_HTML,full_html=True,include_plotlyjs=True,auto_play=False)
print(f'Archivo HTML autónomo guardado: {ARCHIVO_HTML}')
print('Puedes abrirlo en el navegador sin conexión a Internet.')
# Plotly se muestra dentro de Google Colab, con controles de reproducción.
fig.show(renderer='colab' if 'google.colab' in sys.modules else 'notebook_connected')

In [ ]:
#@title 5. Descargar HTML desde Google Colab (opcional)
try:
    from google.colab import files
    files.download(ARCHIVO_HTML)
except ImportError:
    print(f"El archivo está en la carpeta actual: {ARCHIVO_HTML}")